# **1. QRC Feature Extraction and Tagging**
*Creates metadata registry file: **image_metadata.csv***

This notebook serves as the foundational data ingestion, preprocessing, and feature extraction pipeline for the **QRC-Mosaic** framework. It provides an interactive, widget-based workflow to register, standardise, and analyse a historical corpus of retro-futuristic artworks.

Through a structured 5-stage pipeline, the notebook enables users to:
1. **Register Artwork & Log Provenance:** Ingest raw images, assign unique sequential identifiers (`FV001`), and record detailed source and cultural metadata in a master tracking registry (`image_metadata.csv`).
2. **Standardise & Crop Assets:** Interactively define square crop regions using real-time preview controls, exporting standardised 512 × 512px image assets.
3. **Extract Quantitative Colour Profiles:** Compute CIE Lab colour space statistics (\(L^*, a^*, b^*\)), perceptual metrics (brightness, contrast, saturation), compact 3D histograms, and K-means dominant colour clusters.
4. **Generate Deep Visual Embeddings:** Extract 576-dimensional dense visual feature vectors using a pretrained MobileNetV3-Small backbone to support downstream similarity mapping.
5. **Annotate Semantic Metadata:** Leverage the BLIP Vision-Language Model (`Salesforce/blip-image-captioning-base`) to generate natural language captions, refine descriptions, and assign controlled thematic and future outlook tags.

This multimodal dataset—combining quantitative colour profiles, deep neural embeddings, and verified semantic metadata—prepares the corpus for the subsequent quantum sorting and mosaic generation algorithms.

### 1. Environment Setup & Core Imports

Imports standard Python libraries and Google Colab utilities required for data processing, image manipulation, and UI display.

**Features:**
- **Core Data & Image Libraries:** Imports `pandas`, `numpy`, `cv2` (OpenCV), `PIL`, and `matplotlib`.
- **UI & Colab Utilities:** Imports `ipywidgets` and `IPython.display` for interactive widgets, alongside `google.colab.files` for file interactions.

In [ ]:
import os
import cv2
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime
from google.colab import files

### 1. Image Registration and Provenance Tracking

Creates the core project directory structure (`/images/original` and `/metadata`) and initialises a master tracking registry in `image_metadata.csv`. Prompts the user to upload an artwork, automatically assigns a sequential ID (`FV001`), saves the raw file, and records key provenance information.

**Features:**
- **Directory Setup:** Creates `/images/original` and `/metadata` directories automatically.
- **Sequential ID Generation:** Automatically assigns unique identifiers (`FV001`, `FV002`, etc.).
- **File Upload & Ingestion:** Prompts file upload, saves original image bytes, and reads dimensions (`width` × `height`) and file format.
- **Provenance Logging:** Captures metadata attributes including source, licence, creator, date, country, and cultural context.
- **Master Registry:** Appends new records to `image_metadata.csv` and displays a confirmation overview.


In [ ]:
# ============================================================
# 1. REGISTER
# ============================================================
# Upload an image, assign an FV ID, save the original,
# and record basic provenance in image_metadata.csv.
#
# Semantic description is initially left blank and will be
# populated later during Section 5 (Semantic Tagging).
# ============================================================

import os
import pandas as pd
from PIL import Image
from datetime import datetime
from google.colab import files


# ============================================================
# 1A. CREATE PROJECT DIRECTORIES
# ============================================================

PROJECT_DIR = "/content/project"

ORIGINAL_DIR = os.path.join(
    PROJECT_DIR,
    "images",
    "original"
)

METADATA_DIR = os.path.join(
    PROJECT_DIR,
    "metadata"
)

# Create directories if they do not already exist
os.makedirs(ORIGINAL_DIR, exist_ok=True)
os.makedirs(METADATA_DIR, exist_ok=True)

METADATA_FILE = os.path.join(
    METADATA_DIR,
    "image_metadata.csv"
)

print("Project directories ready.")
print(f"Original images: {ORIGINAL_DIR}")
print(f"Metadata file:   {METADATA_FILE}")


# ============================================================
# 1B. DEFINE METADATA COLUMNS
# ============================================================

METADATA_COLUMNS = [
    "image_id",
    "original_filename",
    "original_path",
    "original_width",
    "original_height",
    "file_type",

    # Semantic description
    # Populated later during Section 5
    "description",

    # Provenance / source information
    "source",
    "licence",
    "creator",
    "date",
    "creator_country",
    "cultural_context",

    "registered_at"
]


# ============================================================
# 1C. LOAD EXISTING METADATA
# ============================================================

if os.path.exists(METADATA_FILE):

    metadata_df = pd.read_csv(
        METADATA_FILE
    )

    # Make sure all expected columns exist.
    # This allows the schema to evolve without
    # losing existing metadata.
    for column in METADATA_COLUMNS:

        if column not in metadata_df.columns:

            metadata_df[column] = ""

    # Keep columns in the intended order
    metadata_df = metadata_df[
        METADATA_COLUMNS
    ]

    print(
        f"\nExisting metadata loaded: "
        f"{len(metadata_df)} image(s)"
    )

else:

    metadata_df = pd.DataFrame(
        columns=METADATA_COLUMNS
    )

    print("\nNo existing metadata found.")
    print("A new metadata file will be created.")


# ============================================================
# 1D. GENERATE NEXT FV ID
# ============================================================

def get_next_image_id(metadata_df):

    """
    Generate the next available ID:

    FV001
    FV002
    FV003
    ...
    """

    if metadata_df.empty:

        return "FV001"

    existing_ids = (
        metadata_df["image_id"]
        .dropna()
        .astype(str)
    )

    numbers = []

    for image_id in existing_ids:

        if image_id.startswith("FV"):

            try:

                number = int(
                    image_id[2:]
                )

                numbers.append(
                    number
                )

            except ValueError:

                pass

    if numbers:

        next_number = max(numbers) + 1

    else:

        next_number = 1

    return f"FV{next_number:03d}"


# ============================================================
# 1E. UPLOAD IMAGE
# ============================================================

print("\nSelect an image to register.")

uploaded = files.upload()


# ============================================================
# 1F. CHECK UPLOAD
# ============================================================

if not uploaded:

    print("\nNo image was uploaded.")

else:

    # Register the first uploaded file
    original_filename = list(
        uploaded.keys()
    )[0]

    file_bytes = uploaded[
        original_filename
    ]

    print(
        f"\nUploaded: {original_filename}"
    )


    # ========================================================
    # 1G. ASSIGN FV ID
    # ========================================================

    image_id = get_next_image_id(
        metadata_df
    )

    print(
        f"Assigned image ID: {image_id}"
    )


    # ========================================================
    # 1H. SAVE ORIGINAL IMAGE
    # ========================================================

    original_path = os.path.join(
        ORIGINAL_DIR,
        f"{image_id}_{original_filename}"
    )

    with open(
        original_path,
        "wb"
    ) as f:

        f.write(
            file_bytes
        )

    print(
        f"Original saved to:\n"
        f"{original_path}"
    )


    # ========================================================
    # 1I. READ IMAGE INFORMATION
    # ========================================================

    try:

        with Image.open(
            original_path
        ) as img:

            original_width, original_height = (
                img.size
            )

            file_type = img.format

    except Exception as e:

        print(
            f"\nError reading image: {e}"
        )

    else:

        # ====================================================
        # 1J. COLLECT SOURCE INFORMATION
        # ====================================================

        print(
            "\nEnter available source information."
        )

        print(
            "Leave a field blank if unknown.\n"
        )


        source = input(
            "Source URL or description: "
        ).strip()


        licence = input(
            "Licence / usage rights: "
        ).strip()


        creator = input(
            "Creator / photographer / artist: "
        ).strip()


        date = input(
            "Date or year (if known): "
        ).strip()


        creator_country = input(
            "Creator country (if known; "
            "leave blank if not applicable): "
        ).strip()


        cultural_context = input(
            "Cultural/contextual information "
            "(if known): "
        ).strip()


        # ====================================================
        # 1K. CREATE METADATA RECORD
        # ====================================================

        # Description is intentionally blank at registration.
        # It will be generated/reviewed during Section 5.
        new_record = {

            "image_id":
                image_id,

            "original_filename":
                original_filename,

            "original_path":
                original_path,

            "original_width":
                original_width,

            "original_height":
                original_height,

            "file_type":
                file_type,

            "description":
                "",

            "source":
                source,

            "licence":
                licence,

            "creator":
                creator,

            "date":
                date,

            "creator_country":
                creator_country,

            "cultural_context":
                cultural_context,

            "registered_at":
                datetime.now().isoformat(
                    timespec="seconds"
                )
        }


        # ====================================================
        # 1L. ADD RECORD TO DATAFRAME
        # ====================================================

        metadata_df = pd.concat(
            [
                metadata_df,
                pd.DataFrame(
                    [new_record]
                )
            ],
            ignore_index=True
        )


        # ====================================================
        # 1M. SAVE METADATA CSV
        # ====================================================

        metadata_df.to_csv(
            METADATA_FILE,
            index=False
        )


        # ====================================================
        # 1N. CONFIRMATION
        # ====================================================

        print(
            "\n" + "=" * 60
        )

        print(
            "IMAGE REGISTERED SUCCESSFULLY"
        )

        print(
            "=" * 60
        )

        print(
            f"Image ID:       {image_id}"
        )

        print(
            f"Filename:       {original_filename}"
        )

        print(
            f"Dimensions:     "
            f"{original_width} × "
            f"{original_height}px"
        )

        print(
            f"File type:      {file_type}"
        )

        print(
            f"Original saved: "
            f"{original_path}"
        )

        print(
            f"Metadata saved: "
            f"{METADATA_FILE}"
        )

        print(
            f"\nImages registered: "
            f"{len(metadata_df)} / 64"
        )


        display(
            metadata_df.tail(1)
        )

### 2. Image Selection and Cropping Interface

Initialises an interactive image selection and square crop interface using slider controls. Allows the user to define exact crop dimensions and coordinates (X and Y), previews the selection in real time, and resizes the confirmed crop to \(512 \times 512\text{ px}\) for standardized model input.

**Features:**
- **Directory Initialisation:** Creates the `/images/cropped` directory for processed outputs.
- **Image Selection:** Provides a dropdown selector generated from registered images in `image_metadata.csv`.
- **Interactive Crop Controls:** Interactive sliders for crop size, X position, and Y position with real-time bounding box visualization via Matplotlib.
- **Image Standardisation:** Crops the selected region, resizes it to \(512 \times 512\text{ px}\) (Lanczos resampling), and exports JPEG files named by ID (e.g. `FV001.jpg`).
- **Metadata Logging:** Records crop parameters (`crop_x`, `crop_y`, `crop_size`, `processed_size`, and `cropped_path`) back to `image_metadata.csv`.

In [ ]:
# ============================================================
# 2. CROP / STANDARDISE
# ============================================================
# Purpose:
#   - Select a registered image
#   - Display the original image
#   - Allow the user to adjust a square crop
#   - Preview the crop
#   - Confirm the crop
#   - Resize the confirmed crop to 512 x 512 px
#   - Save as images/cropped/FV001.jpg
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output


# ============================================================
# 2A. CREATE CROPPED IMAGE DIRECTORY
# ============================================================

CROPPED_DIR = os.path.join(
    PROJECT_DIR,
    "images",
    "cropped"
)

os.makedirs(
    CROPPED_DIR,
    exist_ok=True
)

PROCESSED_SIZE = 512

print("Crop directory ready:")
print(CROPPED_DIR)


# ============================================================
# 2B. CHECK REGISTERED IMAGES
# ============================================================

if metadata_df.empty:

    print(
        "No registered images found. "
        "Run Section 1. REGISTER first."
    )

else:

    print(
        f"{len(metadata_df)} registered image(s) available."
    )


# ============================================================
# 2C. SELECT IMAGE TO CROP
# ============================================================

if not metadata_df.empty:

    image_options = [
        (
            f"{row.image_id} — {row.original_filename}",
            row.image_id
        )
        for _, row in metadata_df.iterrows()
    ]

    image_selector = widgets.Dropdown(
        options=image_options,
        description="Image:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="600px")
    )

    display(image_selector)


# ============================================================
# 2D. CROP UI FUNCTION
# ============================================================

def create_crop_interface(image_id):

    clear_output(wait=True)

    # --------------------------------------------------------
    # Get metadata record
    # --------------------------------------------------------

    record = metadata_df[
        metadata_df["image_id"] == image_id
    ]

    if record.empty:

        print(
            f"Could not find metadata for {image_id}."
        )
        return

    record = record.iloc[0]

    original_path = record["original_path"]


    # --------------------------------------------------------
    # Load original image
    # --------------------------------------------------------

    try:

        image = Image.open(
            original_path
        ).convert("RGB")

    except Exception as e:

        print(
            f"Could not open image: {e}"
        )
        return


    image_width, image_height = image.size

    print(
        f"{image_id}: "
        f"{image_width} × {image_height}px"
    )


    # --------------------------------------------------------
    # Maximum possible square crop
    # --------------------------------------------------------

    max_crop_size = min(
        image_width,
        image_height
    )


    # --------------------------------------------------------
    # Initial crop settings
    # --------------------------------------------------------

    initial_crop_size = max_crop_size

    initial_x = int(
        (image_width - initial_crop_size) / 2
    )

    initial_y = int(
        (image_height - initial_crop_size) / 2
    )


    # --------------------------------------------------------
    # Create sliders
    # --------------------------------------------------------

    crop_size_slider = widgets.IntSlider(
        value=initial_crop_size,
        min=100,
        max=max_crop_size,
        step=1,
        description="Crop size:",
        style={"description_width": "100px"},
        layout=widgets.Layout(width="700px"),
        continuous_update=False
    )

    x_slider = widgets.IntSlider(
        value=initial_x,
        min=0,
        max=max(
            0,
            image_width - initial_crop_size
        ),
        step=1,
        description="X position:",
        style={"description_width": "100px"},
        layout=widgets.Layout(width="700px"),
        continuous_update=False
    )

    y_slider = widgets.IntSlider(
        value=initial_y,
        min=0,
        max=max(
            0,
            image_height - initial_crop_size
        ),
        step=1,
        description="Y position:",
        style={"description_width": "100px"},
        layout=widgets.Layout(width="700px"),
        continuous_update=False
    )


    # --------------------------------------------------------
    # Preview output
    # --------------------------------------------------------

    preview_output = widgets.Output()


    # --------------------------------------------------------
    # Update slider ranges when crop size changes
    # --------------------------------------------------------

    def update_position_ranges(change):

        crop_size = change["new"]

        x_slider.max = max(
            0,
            image_width - crop_size
        )

        y_slider.max = max(
            0,
            image_height - crop_size
        )

        # Keep existing positions valid
        if x_slider.value > x_slider.max:
            x_slider.value = x_slider.max

        if y_slider.value > y_slider.max:
            y_slider.value = y_slider.max


    crop_size_slider.observe(
        update_position_ranges,
        names="value"
    )


    # --------------------------------------------------------
    # Draw preview
    # --------------------------------------------------------

    def update_preview(*args):

        crop_size = crop_size_slider.value
        x = x_slider.value
        y = y_slider.value

        cropped = image.crop(
            (
                x,
                y,
                x + crop_size,
                y + crop_size
            )
        )

        with preview_output:

            clear_output(wait=True)

            fig, axes = plt.subplots(
                1,
                2,
                figsize=(12, 6)
            )

            # Original image
            axes[0].imshow(image)
            axes[0].set_title(
                f"Original — {image_width} × {image_height}px"
            )
            axes[0].axis("off")

            # Draw crop boundary
            rectangle = plt.Rectangle(
                (x, y),
                crop_size,
                crop_size,
                fill=False,
                linewidth=2
            )

            axes[0].add_patch(
                rectangle
            )

            # Cropped preview
            axes[1].imshow(cropped)
            axes[1].set_title(
                f"Square crop — {crop_size} × {crop_size}px"
            )
            axes[1].axis("off")

            plt.tight_layout()
            plt.show()


    # --------------------------------------------------------
    # Connect sliders to preview
    # --------------------------------------------------------

    crop_size_slider.observe(
        update_preview,
        names="value"
    )

    x_slider.observe(
        update_preview,
        names="value"
    )

    y_slider.observe(
        update_preview,
        names="value"
    )


    # --------------------------------------------------------
    # Confirm button
    # --------------------------------------------------------

    confirm_button = widgets.Button(
        description="Confirm crop",
        button_style="success",
        icon="check"
    )

    status_output = widgets.Output()


    # --------------------------------------------------------
    # Save confirmed crop
    # --------------------------------------------------------

    def confirm_crop(button):

        crop_size = crop_size_slider.value
        x = x_slider.value
        y = y_slider.value

        cropped = image.crop(
            (
                x,
                y,
                x + crop_size,
                y + crop_size
            )
        )

        # Resize to common processing size
        processed = cropped.resize(
            (
                PROCESSED_SIZE,
                PROCESSED_SIZE
            ),
            Image.Resampling.LANCZOS
        )

        # Save using stable FV ID
        cropped_filename = (
            f"{image_id}.jpg"
        )

        cropped_path = os.path.join(
            CROPPED_DIR,
            cropped_filename
        )

        processed.save(
            cropped_path,
            "JPEG",
            quality=95
        )


        # ----------------------------------------------------
        # Add crop information to metadata
        # ----------------------------------------------------

        metadata_df.loc[
            metadata_df["image_id"] == image_id,
            "crop_x"
        ] = x

        metadata_df.loc[
            metadata_df["image_id"] == image_id,
            "crop_y"
        ] = y

        metadata_df.loc[
            metadata_df["image_id"] == image_id,
            "crop_size"
        ] = crop_size

        metadata_df.loc[
            metadata_df["image_id"] == image_id,
            "processed_size"
        ] = PROCESSED_SIZE

        metadata_df.loc[
            metadata_df["image_id"] == image_id,
            "cropped_path"
        ] = cropped_path


        # ----------------------------------------------------
        # Save metadata
        # ----------------------------------------------------

        metadata_df.to_csv(
            METADATA_FILE,
            index=False
        )


        # ----------------------------------------------------
        # Confirmation message
        # ----------------------------------------------------

        with status_output:

            clear_output(wait=True)

            print(
                "✓ Crop confirmed and saved."
            )

            print(
                f"\nImage ID:       {image_id}"
            )

            print(
                f"Crop:           "
                f"{crop_size} × {crop_size}px"
            )

            print(
                f"Position:       "
                f"X={x}, Y={y}"
            )

            print(
                f"Processed size: "
                f"{PROCESSED_SIZE} × "
                f"{PROCESSED_SIZE}px"
            )

            print(
                f"\nSaved to:\n{cropped_path}"
            )


    confirm_button.on_click(
        confirm_crop
    )


    # --------------------------------------------------------
    # Display interface
    # --------------------------------------------------------

    print(
        "\nAdjust the square crop using the sliders."
    )

    print(
        "The original image is never modified."
    )

    display(
        crop_size_slider,
        x_slider,
        y_slider,
        preview_output,
        confirm_button,
        status_output
    )

    # Initial preview
    update_preview()


# ============================================================
# 2E. LAUNCH CROP INTERFACE
# ============================================================

if not metadata_df.empty:

    display(
        widgets.Button(
            description="Load selected image",
            button_style="primary"
        )
    )

    load_button = widgets.Button(
        description="Open crop interface",
        button_style="primary",
        icon="crop"
    )

    display(load_button)


    def load_selected_image(button):

        create_crop_interface(
            image_selector.value
        )


    load_button.on_click(
        load_selected_image
    )

FV001: 2607 × 3886px

Adjust the square crop using the sliders.
The original image is never modified.


IntSlider(value=2607, continuous_update=False, description='Crop size:', layout=Layout(width='700px'), max=260…

IntSlider(value=0, continuous_update=False, description='X position:', layout=Layout(width='700px'), max=0, st…

IntSlider(value=639, continuous_update=False, description='Y position:', layout=Layout(width='700px'), max=127…

Output()

Button(button_style='success', description='Confirm crop', icon='check', style=ButtonStyle())

Output()

### 3. Colour Feature Extraction and Profiling

Extracts comprehensive quantitative colour metrics and profiles from the processed \(512 \times 512\text{ px}\) cropped image without modifying the original asset.

**Features:**
- **Directory Setup:** Initialises output directories for `/features/colour` and `/features/visual`.
- **CIE Lab Profiling:** Calculates mean and standard deviation for \(L^*\), \(a^*\), and \(b^*\) colour space channels.
- **Perceptual Metrics:** Computes overall image brightness, contrast, and saturation scores.
- **Colour Histogram:** Generates a compact \(8 \times 8 \times 8\) CIE Lab histogram saved as a `.npy` file.
- **Dominant Colour Analysis:** Identifies top 5 dominant RGB colours and their spatial proportions using K-means clustering, saved as a `.json` file.
- **Metadata Logging:** Updates `image_metadata.csv` with extracted colour attributes and file paths.

In [ ]:
# ============================================================
# 3. COLOUR + 4. VISUAL FEATURE EXTRACTION
# ============================================================
#
# 3. COLOUR
#   - CIE Lab mean / standard deviation
#   - Brightness
#   - Contrast
#   - Saturation
#   - Compact Lab histogram
#   - Dominant colours
#
# 4. VISUAL
#   - Pretrained MobileNetV3-Small
#   - Visual embedding
#   - Embedding saved separately as .npy
#
# The processed 512 x 512 crop is used for analysis.
# The original image is never modified.
# ============================================================


# ============================================================
# 3A. INSTALL / IMPORT REQUIRED PACKAGES
# ============================================================

!pip -q install scikit-image scikit-learn


import os
import json
import numpy as np
import pandas as pd

from PIL import Image

from skimage.color import rgb2lab

from sklearn.cluster import KMeans

import torch
import torch.nn as nn

from torchvision import models
from torchvision.models import MobileNet_V3_Small_Weights

from IPython.display import display


# ============================================================
# 3B. CREATE FEATURE DIRECTORIES
# ============================================================

FEATURES_DIR = os.path.join(
    PROJECT_DIR,
    "features"
)

COLOUR_DIR = os.path.join(
    FEATURES_DIR,
    "colour"
)

VISUAL_DIR = os.path.join(
    FEATURES_DIR,
    "visual"
)

os.makedirs(
    COLOUR_DIR,
    exist_ok=True
)

os.makedirs(
    VISUAL_DIR,
    exist_ok=True
)

print("Feature directories ready:")
print(f"Colour features: {COLOUR_DIR}")
print(f"Visual features: {VISUAL_DIR}")


# ============================================================
# 3C. LOAD PROCESSED IMAGE
# ============================================================

def load_processed_image(image_id):
    """
    Load the processed 512 x 512 image for an FV ID.
    """

    image_row = metadata_df[
        metadata_df["image_id"] == image_id
    ]

    if image_row.empty:

        raise ValueError(
            f"{image_id} not found in metadata_df."
        )

    row = image_row.iloc[0]

    # Use the recorded cropped path when available
    if (
        "cropped_path" in metadata_df.columns
        and pd.notna(row.get("cropped_path"))
        and str(row.get("cropped_path")).strip() != ""
    ):

        image_path = str(
            row["cropped_path"]
        )

    else:

        image_path = os.path.join(
            CROPPED_DIR,
            f"{image_id}.jpg"
        )

    if not os.path.exists(image_path):

        raise FileNotFoundError(
            f"Processed image not found:\n{image_path}"
        )

    image = Image.open(
        image_path
    ).convert("RGB")

    return image, image_path


# ============================================================
# 3D. CIE LAB COLOUR PROFILE
# ============================================================

def calculate_lab_profile(image):
    """
    Calculate CIE Lab colour statistics.

    Returns:
        mean_L
        mean_a
        mean_b
        std_L
        std_a
        std_b
    """

    rgb = np.asarray(
        image
    ).astype(np.float32) / 255.0

    lab = rgb2lab(
        rgb
    )

    return {
        "mean_L": float(
            np.mean(lab[:, :, 0])
        ),

        "mean_a": float(
            np.mean(lab[:, :, 1])
        ),

        "mean_b": float(
            np.mean(lab[:, :, 2])
        ),

        "std_L": float(
            np.std(lab[:, :, 0])
        ),

        "std_a": float(
            np.std(lab[:, :, 1])
        ),

        "std_b": float(
            np.std(lab[:, :, 2])
        )
    }


# ============================================================
# 3E. BASIC COLOUR METRICS
# ============================================================

def calculate_basic_colour_metrics(image):

    rgb = np.asarray(
        image
    ).astype(np.float32) / 255.0

    # --------------------------------------------------------
    # Brightness
    # --------------------------------------------------------

    brightness = float(
        np.mean(rgb)
    )

    # --------------------------------------------------------
    # Contrast
    # --------------------------------------------------------

    grayscale = (
        0.299 * rgb[:, :, 0]
        + 0.587 * rgb[:, :, 1]
        + 0.114 * rgb[:, :, 2]
    )

    contrast = float(
        np.std(grayscale)
    )

    # --------------------------------------------------------
    # Saturation
    # --------------------------------------------------------

    hsv = np.asarray(
        image.convert("HSV")
    ).astype(np.float32)

    saturation = float(
        np.mean(hsv[:, :, 1]) / 255.0
    )

    return {
        "brightness": brightness,
        "contrast": contrast,
        "saturation": saturation
    }


# ============================================================
# 3F. COMPACT LAB HISTOGRAM
# ============================================================

def calculate_lab_histogram(
    image,
    bins_L=8,
    bins_a=8,
    bins_b=8
):
    """
    Create a compact 8 x 8 x 8 Lab histogram.

    This is saved separately rather than storing the full
    histogram inside image_metadata.csv.
    """

    rgb = np.asarray(
        image
    ).astype(np.float32) / 255.0

    lab = rgb2lab(
        rgb
    )

    L = lab[:, :, 0]
    a = lab[:, :, 1]
    b = lab[:, :, 2]

    pixels = np.column_stack(
        [
            L.ravel(),
            a.ravel(),
            b.ravel()
        ]
    )

    histogram, _ = np.histogramdd(
        pixels,
        bins=[
            bins_L,
            bins_a,
            bins_b
        ],
        range=[
            (0, 100),
            (-128, 127),
            (-128, 127)
        ]
    )

    histogram = histogram.astype(
        np.float32
    )

    histogram /= (
        histogram.sum() + 1e-12
    )

    return histogram


# ============================================================
# 3G. DOMINANT COLOURS
# ============================================================

def calculate_dominant_colours(
    image,
    n_colours=5,
    sample_size=10000
):
    """
    Identify the five dominant RGB colours using K-means.

    Returns a list containing RGB values and their proportions.
    """

    rgb = np.asarray(
        image
    )

    pixels = rgb.reshape(
        -1,
        3
    )

    # Reduce the number of pixels used by K-means
    if len(pixels) > sample_size:

        rng = np.random.default_rng(
            42
        )

        indices = rng.choice(
            len(pixels),
            size=sample_size,
            replace=False
        )

        pixels = pixels[
            indices
        ]

    kmeans = KMeans(
        n_clusters=n_colours,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(
        pixels
    )

    centres = kmeans.cluster_centers_

    counts = np.bincount(
        labels,
        minlength=n_colours
    )

    proportions = (
        counts / counts.sum()
    )

    # Largest colour proportion first
    order = np.argsort(
        proportions
    )[::-1]

    dominant_colours = []

    for index in order:

        rgb_colour = np.clip(
            np.round(
                centres[index]
            ),
            0,
            255
        ).astype(int)

        dominant_colours.append(
            {
                "rgb": rgb_colour.tolist(),

                "proportion": float(
                    proportions[index]
                )
            }
        )

    return dominant_colours


# ============================================================
# 3H. COLOUR FEATURE EXTRACTION
# ============================================================

def extract_colour_features(image_id):

    image, image_path = load_processed_image(
        image_id
    )

    # --------------------------------------------------------
    # Lab profile
    # --------------------------------------------------------

    lab_profile = calculate_lab_profile(
        image
    )

    # --------------------------------------------------------
    # Basic colour metrics
    # --------------------------------------------------------

    basic_metrics = calculate_basic_colour_metrics(
        image
    )

    # --------------------------------------------------------
    # Lab histogram
    # --------------------------------------------------------

    lab_histogram = calculate_lab_histogram(
        image
    )

    histogram_path = os.path.join(
        COLOUR_DIR,
        f"{image_id}_lab_histogram.npy"
    )

    np.save(
        histogram_path,
        lab_histogram
    )

    # --------------------------------------------------------
    # Dominant colours
    # --------------------------------------------------------

    dominant_colours = calculate_dominant_colours(
        image
    )

    dominant_path = os.path.join(
        COLOUR_DIR,
        f"{image_id}_dominant_colours.json"
    )

    with open(
        dominant_path,
        "w"
    ) as f:

        json.dump(
            dominant_colours,
            f,
            indent=2
        )

    # --------------------------------------------------------
    # Build feature record
    # --------------------------------------------------------

    feature_record = {}

    feature_record.update(
        lab_profile
    )

    feature_record.update(
        basic_metrics
    )

    feature_record[
        "colour_histogram_path"
    ] = histogram_path

    feature_record[
        "dominant_colour_path"
    ] = dominant_path

    # --------------------------------------------------------
    # Store dominant colours as individual fields
    # --------------------------------------------------------

    for i, colour in enumerate(
        dominant_colours,
        start=1
    ):

        feature_record[
            f"dominant_colour_{i}"
        ] = str(
            colour["rgb"]
        )

        feature_record[
            f"dominant_colour_{i}_proportion"
        ] = float(
            colour["proportion"]
        )


    # --------------------------------------------------------
    # Update metadata DataFrame
    # --------------------------------------------------------

    for column, value in feature_record.items():

        if column not in metadata_df.columns:

            # Strings/lists need object dtype.
            # Numeric features use float64.
            if isinstance(
                value,
                (str, list)
            ):

                metadata_df[column] = pd.Series(
                    index=metadata_df.index,
                    dtype="object"
                )

            else:

                metadata_df[column] = pd.Series(
                    index=metadata_df.index,
                    dtype="float64"
                )

        metadata_df.loc[
            metadata_df["image_id"] == image_id,
            column
        ] = value


    # --------------------------------------------------------
    # Save metadata
    # --------------------------------------------------------

    metadata_df.to_csv(
        METADATA_FILE,
        index=False
    )

    return feature_record




Feature directories ready:
Colour features: /content/project/features/colour
Visual features: /content/project/features/visual

Loading pretrained MobileNetV3-Small...
Visual model ready.


Dropdown(description='Image:', layout=Layout(width='600px'), options=(('FV001 — grand_tour.jpg', 'FV001'),), s…

Button(button_style='primary', description='Extract features', icon='bar-chart', style=ButtonStyle())


Processing FV001...
1/2  Extracting colour features...
     ✓ Colour profile complete.
2/2  Extracting visual embedding...
     ✓ Visual embedding complete.

FEATURE EXTRACTION COMPLETE — FV001
Lab mean: L=51.49, a=1.54, b=13.91
Brightness: 0.445
Contrast: 0.254
Saturation: 0.557
Visual embedding: 576 dimensions

Feature files saved.
Metadata updated:
/content/project/metadata/image_metadata.csv


### 4. Deep Visual Feature Extraction and Pipeline Execution

Extracts deep visual embeddings using a pretrained MobileNetV3-Small backbone and executes the combined feature extraction pipeline (colour metrics, histograms, dominant colours, and visual embeddings) for selected images.

**Features:**
- **Pretrained Vision Backbone:** Initialises a MobileNetV3-Small model with default PyTorch weights, replacing the classification head with `nn.Identity()` to extract feature representations.
- **Visual Embedding Generation:** Transforms the \(512 \times 512\text{ px}\) image, computes a 576-dimensional dense feature vector, and saves it as a `.npy` file in `/features/visual/`.
- **Unified Feature Pipeline:** Provides `process_features()` to run both CIE Lab colour profiling (Cell 3) and deep visual embedding in a single execution step.
- **Interactive UI:** Features an interactive dropdown and button interface to select registered images and trigger feature extraction.
- **Metadata Logging:** Updates `image_metadata.csv` with `visual_embedding_path` and `visual_embedding_dimension` alongside summary logs.

In [ ]:
# ============================================================
# 4A. LOAD PRETRAINED VISION MODEL
# ============================================================

print(
    "\nLoading pretrained MobileNetV3-Small..."
)

weights = (
    MobileNet_V3_Small_Weights.DEFAULT
)

visual_model = models.mobilenet_v3_small(
    weights=weights
)

# Remove the classification layer.
# The remaining network produces the visual representation.
visual_model.classifier = nn.Identity()

visual_model.eval()

visual_preprocess = (
    weights.transforms()
)

print(
    "Visual model ready."
)


# ============================================================
# 4B. EXTRACT VISUAL EMBEDDING
# ============================================================

def extract_visual_embedding(image_id):

    image, image_path = load_processed_image(
        image_id
    )

    # --------------------------------------------------------
    # Prepare image
    # --------------------------------------------------------

    input_tensor = visual_preprocess(
        image
    ).unsqueeze(0)

    # --------------------------------------------------------
    # Generate embedding
    # --------------------------------------------------------

    with torch.no_grad():

        embedding = visual_model(
            input_tensor
        )

    embedding = (
        embedding
        .squeeze()
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    # --------------------------------------------------------
    # Save embedding separately
    # --------------------------------------------------------

    embedding_path = os.path.join(
        VISUAL_DIR,
        f"{image_id}_embedding.npy"
    )

    np.save(
        embedding_path,
        embedding
    )


    # --------------------------------------------------------
    # Create metadata columns if required
    # --------------------------------------------------------

    if "visual_embedding_path" not in metadata_df.columns:

        metadata_df["visual_embedding_path"] = (
            pd.Series(
                index=metadata_df.index,
                dtype="object"
            )
        )

    if "visual_embedding_dimension" not in metadata_df.columns:

        metadata_df["visual_embedding_dimension"] = (
            pd.Series(
                index=metadata_df.index,
                dtype="float64"
            )
        )


    # --------------------------------------------------------
    # Update metadata
    # --------------------------------------------------------

    metadata_df.loc[
        metadata_df["image_id"] == image_id,
        "visual_embedding_path"
    ] = embedding_path

    metadata_df.loc[
        metadata_df["image_id"] == image_id,
        "visual_embedding_dimension"
    ] = len(embedding)


    # --------------------------------------------------------
    # Save metadata
    # --------------------------------------------------------

    metadata_df.to_csv(
        METADATA_FILE,
        index=False
    )

    return embedding


# ============================================================
# 4C. PROCESS COLOUR + VISUAL FEATURES
# ============================================================

def process_features(image_id):

    print(
        f"\nProcessing {image_id}..."
    )

    # --------------------------------------------------------
    # Colour
    # --------------------------------------------------------

    print(
        "1/2  Extracting colour features..."
    )

    colour_features = extract_colour_features(
        image_id
    )

    print(
        "     ✓ Colour profile complete."
    )


    # --------------------------------------------------------
    # Visual
    # --------------------------------------------------------

    print(
        "2/2  Extracting visual embedding..."
    )

    embedding = extract_visual_embedding(
        image_id
    )

    print(
        "     ✓ Visual embedding complete."
    )


    # --------------------------------------------------------
    # Final summary
    # --------------------------------------------------------

    print(
        "\n" + "=" * 60
    )

    print(
        f"FEATURE EXTRACTION COMPLETE — {image_id}"
    )

    print(
        "=" * 60
    )

    print(
        f"Lab mean: "
        f"L={colour_features['mean_L']:.2f}, "
        f"a={colour_features['mean_a']:.2f}, "
        f"b={colour_features['mean_b']:.2f}"
    )

    print(
        f"Brightness: "
        f"{colour_features['brightness']:.3f}"
    )

    print(
        f"Contrast: "
        f"{colour_features['contrast']:.3f}"
    )

    print(
        f"Saturation: "
        f"{colour_features['saturation']:.3f}"
    )

    print(
        f"Visual embedding: "
        f"{len(embedding)} dimensions"
    )

    print(
        "\nFeature files saved."
    )

    print(
        f"Metadata updated:\n{METADATA_FILE}"
    )


# ============================================================
# 4D. IMAGE SELECTION
# ============================================================

if not metadata_df.empty:

    feature_options = [
        (
            f"{row.image_id} — "
            f"{row.original_filename}",
            row.image_id
        )
        for _, row in metadata_df.iterrows()
    ]

    feature_selector = widgets.Dropdown(
        options=feature_options,
        description="Image:",
        style={
            "description_width": "initial"
        },
        layout=widgets.Layout(
            width="600px"
        )
    )

    process_button = widgets.Button(
        description="Extract features",
        button_style="primary",
        icon="bar-chart"
    )

    display(
        feature_selector,
        process_button
    )


    def run_feature_extraction(button):

        process_features(
            feature_selector.value
        )


    process_button.on_click(
        run_feature_extraction
    )

else:

    print(
        "No registered images found."
    )

    print(
        "Run Section 1. REGISTER first."
    )

### 5. Semantic Tagging and AI Image Captioning

Initialises an interactive annotation interface utilizing the BLIP vision-language model (`Salesforce/blip-image-captioning-base`) to generate descriptive captions, review/edit text, and assign controlled thematic and outlook metadata.

**Features:**
- **Automated BLIP Captioning:** Uses a pretrained BLIP model to generate short, natural language image captions.
- **Interactive Review & Edit:** Provides a text area widget allowing users to inspect and refine generated captions prior to saving.
- **Thematic Tagging:** Categorises images using a controlled vocabulary of 12 themes (`future_city`, `AI_robotics`, `space`, `transport`, etc.).
- **Future Outlook Assessment:** Assigns a contextual outlook tag (`optimistic`, `neutral`, `dystopian`, `ambiguous`).
- **Metadata Registry Export:** Saves verified descriptions, themes, and outlooks back to `image_metadata.csv`.

In [ ]:
# ============================================================
# 5. SEMANTIC TAGGING
# ============================================================
# Generate an image caption using BLIP, then allow the user
# to review/edit the caption and assign:
#
#   - Theme
#   - Future outlook
#
# The final semantic metadata is saved to image_metadata.csv.
# ============================================================


# ============================================================
# 5A. INSTALL / IMPORT IMAGE CAPTIONING LIBRARIES
# ============================================================

!pip -q install transformers sentencepiece


import os
import torch
import pandas as pd

from PIL import Image

import ipywidgets as widgets

from IPython.display import (
    display,
    clear_output
)

from transformers import (
    BlipProcessor,
    BlipForConditionalGeneration
)


# ============================================================
# 5B. PROJECT PATHS
# ============================================================

PROJECT_DIR = "/content/project"

METADATA_DIR = os.path.join(
    PROJECT_DIR,
    "metadata"
)

METADATA_FILE = os.path.join(
    METADATA_DIR,
    "image_metadata.csv"
)

CROPPED_DIR = os.path.join(
    PROJECT_DIR,
    "images",
    "cropped"
)


if not os.path.exists(METADATA_FILE):

    raise FileNotFoundError(
        "Metadata file not found:\n"
        f"{METADATA_FILE}\n\n"
        "Run Section 1 first."
    )


metadata_df = pd.read_csv(
    METADATA_FILE
)


print(
    f"Metadata loaded: "
    f"{len(metadata_df)} image(s)"
)


# ============================================================
# 5C. ENSURE SEMANTIC COLUMNS EXIST
# ============================================================

# Ensure semantic metadata columns exist with the correct dtype
semantic_columns = {
    "description": "object",
    "theme": "object",
    "future_outlook": "object"
}

for column, dtype in semantic_columns.items():
    if column not in metadata_df.columns:
        metadata_df[column] = pd.Series(
            index=metadata_df.index,
            dtype=dtype
        )
    else:
        metadata_df[column] = metadata_df[column].astype(dtype)

# ============================================================
# 5D. CONTROLLED VOCABULARIES
# ============================================================

THEME_OPTIONS = [
    "future_city",
    "AI_robotics",
    "space",
    "transport",
    "future_home",
    "healthcare",
    "climate",
    "education",
    "work",
    "human_machine",
    "energy",
    "other"
]


OUTLOOK_OPTIONS = [
    "optimistic",
    "neutral",
    "dystopian",
    "ambiguous"
]


# ============================================================
# 5E. SELECT COMPUTE DEVICE
# ============================================================

if torch.cuda.is_available():

    device = torch.device("cuda")

    print(
        "GPU detected:"
    )

    print(
        torch.cuda.get_device_name(0)
    )

else:

    device = torch.device("cpu")

    print(
        "WARNING: No GPU detected."
    )

    print(
        "BLIP caption generation will be considerably slower."
    )


# ============================================================
# 5F. LOAD BLIP
# ============================================================

BLIP_MODEL_NAME = (
    "Salesforce/blip-image-captioning-base"
)


print(
    "\nLoading BLIP model..."
)

processor = BlipProcessor.from_pretrained(
    BLIP_MODEL_NAME
)


model = BlipForConditionalGeneration.from_pretrained(
    BLIP_MODEL_NAME
)


model = model.to(device)

model.eval()


print(
    "BLIP model loaded successfully."
)


# ============================================================
# 5G. LOAD PROCESSED IMAGE
# ============================================================

def load_processed_image(image_id):

    row = metadata_df[
        metadata_df["image_id"] == image_id
    ]

    if row.empty:

        raise ValueError(
            f"Image ID not found: {image_id}"
        )

    row = row.iloc[0]


    # Prefer the path recorded during Section 2
    if (
        "cropped_path" in row.index
        and pd.notna(row["cropped_path"])
        and str(row["cropped_path"]).strip()
    ):

        image_path = str(
            row["cropped_path"]
        )

    else:

        image_path = os.path.join(
            CROPPED_DIR,
            f"{image_id}.jpg"
        )


    if not os.path.exists(image_path):

        raise FileNotFoundError(
            "Processed image not found:\n"
            f"{image_path}\n\n"
            "Run Section 2 first."
        )


    image = Image.open(
        image_path
    ).convert("RGB")


    return image, image_path


# ============================================================
# 5H. GENERATE CAPTION
# ============================================================

def generate_caption(image):

    """
    Generate a short image caption using BLIP.

    Uses greedy decoding rather than beam search to make
    interactive generation substantially faster.
    """

    inputs = processor(
        images=image,
        return_tensors="pt"
    )


    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }


    with torch.inference_mode():

        output = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False,
            num_beams=1
        )


    caption = processor.decode(
        output[0],
        skip_special_tokens=True
    )


    return caption.strip()


# ============================================================
# 5I. USER INTERFACE
# ============================================================

image_selector = widgets.Dropdown(
    options=list(
        metadata_df["image_id"].astype(str)
    ),
    description="Image:",
    layout=widgets.Layout(
        width="350px"
    )
)


generate_button = widgets.Button(
    description="Generate caption",
    button_style="primary",
    layout=widgets.Layout(
        width="200px"
    )
)


save_button = widgets.Button(
    description="Save metadata",
    button_style="success",
    layout=widgets.Layout(
        width="200px"
    )
)


caption_box = widgets.Textarea(
    description="Caption:",
    placeholder=(
        "Generated caption will appear here. "
        "Review or edit it before saving."
    ),
    layout=widgets.Layout(
        width="800px",
        height="100px"
    )
)


theme_selector = widgets.Dropdown(
    options=[
        ("Select theme...", "")
    ] + [
        (theme, theme)
        for theme in THEME_OPTIONS
    ],
    description="Theme:",
    layout=widgets.Layout(
        width="450px"
    )
)


outlook_selector = widgets.Dropdown(
    options=[
        ("Select outlook...", "")
    ] + [
        (outlook, outlook)
        for outlook in OUTLOOK_OPTIONS
    ],
    description="Outlook:",
    layout=widgets.Layout(
        width="450px"
    )
)


output_area = widgets.Output(
    layout=widgets.Layout(
        width="900px"
    )
)


# ============================================================
# 5J. DISPLAY IMAGE
# ============================================================

def display_selected_image(image_id):

    with output_area:

        clear_output(
            wait=True
        )

        try:

            image, image_path = (
                load_processed_image(
                    image_id
                )
            )

            display(
                image
            )

            print(
                f"Image ID: {image_id}"
            )

            print(
                f"Processed image: {image_path}"
            )

        except Exception as e:

            print(
                f"Error loading image: {e}"
            )


# ============================================================
# 5K. IMAGE SELECTION
# ============================================================

def on_image_selected(change):

    if change["name"] != "value":
        return


    image_id = change["new"]


    caption_box.value = ""

    theme_selector.value = ""

    outlook_selector.value = ""


    display_selected_image(
        image_id
    )


image_selector.observe(
    on_image_selected,
    names="value"
)


# ============================================================
# 5L. GENERATE CAPTION
# ============================================================

def on_generate_caption(button):

    image_id = image_selector.value


    generate_button.disabled = True


    with output_area:

        clear_output(
            wait=True
        )

        print(
            f"Generating caption for {image_id}..."
        )

        print(
            "Please wait..."
        )


        try:

            image, image_path = (
                load_processed_image(
                    image_id
                )
            )


            display(
                image
            )


            caption = generate_caption(
                image
            )


            caption_box.value = caption


            print(
                "\nBLIP-generated caption:"
            )

            print(
                caption
            )

            print(
                "\nReview or edit the caption "
                "before saving."
            )


        except Exception as e:

            print(
                "\nCaption generation failed."
            )

            print(
                f"Error: {type(e).__name__}: {e}"
            )


        finally:

            generate_button.disabled = False


generate_button.on_click(
    on_generate_caption
)


# ============================================================
# 5M. SAVE SEMANTIC METADATA
# ============================================================

def on_save_metadata(button):

    image_id = image_selector.value

    final_description = (
        caption_box.value.strip()
    )

    theme = (
        theme_selector.value
    )

    future_outlook = (
        outlook_selector.value
    )


    if not final_description:

        with output_area:

            print(
                "Please enter or review a caption "
                "before saving."
            )

        return


    if not theme:

        with output_area:

            print(
                "Please select a theme."
            )

        return


    if not future_outlook:

        with output_area:

            print(
                "Please select a future outlook."
            )

        return


    # --------------------------------------------------------
    # Update metadata
    # --------------------------------------------------------

    metadata_df.loc[
        metadata_df["image_id"] == image_id,
        "description"
    ] = final_description


    metadata_df.loc[
        metadata_df["image_id"] == image_id,
        "theme"
    ] = theme


    metadata_df.loc[
        metadata_df["image_id"] == image_id,
        "future_outlook"
    ] = future_outlook


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    metadata_df.to_csv(
        METADATA_FILE,
        index=False
    )


    # --------------------------------------------------------
    # Confirmation
    # --------------------------------------------------------

    with output_area:

        clear_output(
            wait=True
        )

        print(
            "=" * 60
        )

        print(
            "SEMANTIC METADATA SAVED"
        )

        print(
            "=" * 60
        )

        print(
            f"Image ID:       {image_id}"
        )

        print(
            f"Theme:          {theme}"
        )

        print(
            f"Future outlook: {future_outlook}"
        )

        print(
            f"Description:    {final_description}"
        )

        print(
            "\nMetadata saved to:"
        )

        print(
            METADATA_FILE
        )


save_button.on_click(
    on_save_metadata
)


# ============================================================
# 5N. DISPLAY INTERFACE
# ============================================================

display(
    image_selector
)

display(
    generate_button
)

display(
    caption_box
)

display(
    theme_selector
)

display(
    outlook_selector
)

display(
    save_button
)

display(
    output_area
)


# ============================================================
# 5O. DISPLAY FIRST IMAGE
# ============================================================

if len(metadata_df) > 0:

    display_selected_image(
        metadata_df.iloc[0]["image_id"]
    )

Metadata loaded: 2 image(s)
BLIP caption generation will be considerably slower.

Loading BLIP model...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

BLIP model loaded successfully.


Dropdown(description='Image:', layout=Layout(width='350px'), options=('FV001', 'FV002'), value='FV001')

Button(button_style='primary', description='Generate caption', layout=Layout(width='200px'), style=ButtonStyle…

...

Dropdown(description='Theme:', layout=Layout(width='450px'), options=(('Select theme...', ''), ('future_city',…

Dropdown(description='Outlook:', layout=Layout(width='450px'), options=(('Select outlook...', ''), ('optimisti…

Button(button_style='success', description='Save metadata', layout=Layout(width='200px'), style=ButtonStyle())

Output(layout=Layout(width='900px'))

/tmp/ipykernel_558/2815533846.py:591: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'JPL travel poster depicting a spacecraft journey through the Solar System, representing a future of space exploration.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  metadata_df.loc[
